## 02d_quality_transform — RCM Intelligence Platform
### Purpose
Transforms CMS Hospital Compare quality measures and HCAHPS
patient satisfaction data from Bronze to Silver.

### What this does
1. Reads hospital_quality_raw and hospital_hcahps_raw from Bronze
2. Filters to key quality measures only (mortality, readmissions, safety)
3. Pivots long format to wide format — one row per hospital
4. Scores each hospital on quality dimensions 0-100
5. Reads HCAHPS and extracts key satisfaction scores
6. Writes to Silver quality tables

### Key measures used
#### Mortality (lower is better)
- MORT_30_AMI  — Heart attack 30-day mortality
- MORT_30_HF   — Heart failure 30-day mortality
- MORT_30_PN   — Pneumonia 30-day mortality
- MORT_30_CABG — CABG surgery 30-day mortality
- MORT_30_STK  — Stroke 30-day mortality

#### Readmissions (lower is better)
- READM_30_AMI — Heart attack 30-day readmission
- READM_30_HF  — Heart failure 30-day readmission
- READM_30_PN  — Pneumonia 30-day readmission
- READM_30_HIP_KNEE — Hip/knee 30-day readmission

#### Patient Safety (lower is better)
- PSI_90 — Patient safety composite

#### HCAHPS (higher is better)
- H_RECMND_DY — % patients who would definitely recommend
- H_OVERALL_STAR_RATING — Overall hospital star rating

### Outputs
- rcm_platform.rcm_silver.hospital_quality_scores
- rcm_platform.rcm_silver.hospital_hcahps_scores

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Silver |
| Run order  | 5 — after 01c_ingest_quality_data |
| Depends on | 01c_ingest_quality_data |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
import builtins
round = builtins.round

from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, when, expr, avg as spark_avg,
    round as spark_round, max as spark_max,
    coalesce, first
)
from pyspark.sql.types import DoubleType

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "02d_quality_transform"

TBL_QUALITY_RAW   = f"{BRONZE_DB}.hospital_quality_raw"
TBL_HCAHPS_RAW    = f"{BRONZE_DB}.hospital_hcahps_raw"
TBL_QUALITY_SILVER = f"{SILVER_DB}.hospital_quality_scores"
TBL_HCAHPS_SILVER  = f"{SILVER_DB}.hospital_hcahps_scores"

print(f"Batch ID : {BATCH_ID}")
print(f"Inputs   : {TBL_QUALITY_RAW}")
print(f"           {TBL_HCAHPS_RAW}")
print(f"Outputs  : {TBL_QUALITY_SILVER}")
print(f"           {TBL_HCAHPS_SILVER}")

In [0]:
# ============================================================
# STEP 1 — INSPECT AVAILABLE QUALITY MEASURES
# ============================================================

print("=" * 55)
print("  INSPECTING QUALITY MEASURES")
print("=" * 55)

df_quality_raw = spark.table(TBL_QUALITY_RAW)

print(f"Total rows : {df_quality_raw.count():,}")

print("\nAll unique measure IDs:")
spark.sql(f"""
    SELECT
        measure_id,
        measure_name,
        COUNT(DISTINCT facility_id) AS hospital_count,
        SUM(CASE WHEN score = 'Not Available' THEN 1 ELSE 0 END) AS not_available_count
    FROM {TBL_QUALITY_RAW}
    GROUP BY measure_id, measure_name
    ORDER BY measure_id
""").show(100, truncate=False)

In [0]:
# ============================================================
# STEP 2 — FILTER AND PIVOT QUALITY MEASURES
# One row per hospital with key measures as columns
# ============================================================

print("=" * 55)
print("  PIVOTING QUALITY MEASURES")
print("=" * 55)

# Key measures we care about
KEY_MEASURES = [
    "MORT_30_AMI",
    "MORT_30_HF",
    "MORT_30_PN",
    "MORT_30_CABG",
    "MORT_30_STK",
    "READM_30_AMI",
    "READM_30_HF",
    "READM_30_PN",
    "READM_30_HIP_KNEE",
    "PSI_90",
    "OP_18b",
    "EDV"
]

measures_str = "', '".join(KEY_MEASURES)

# Filter to key measures and cast score to numeric
df_filtered = spark.sql(f"""
    SELECT
        facility_id                                        AS provider_id,
        facility_name,
        state,
        measure_id,
        measure_name,
        compared_to_national,
        CASE
            WHEN score = 'Not Available' THEN NULL
            WHEN score = 'Not Applicable' THEN NULL
            ELSE TRY_CAST(score AS DOUBLE)
        END                                                AS score_numeric,
        score                                              AS score_raw
    FROM {TBL_QUALITY_RAW}
    WHERE measure_id IN ('{measures_str}')
""")

print(f"Filtered rows : {df_filtered.count():,}")
print(f"Unique hospitals : {df_filtered.select('provider_id').distinct().count():,}")

# Register as temp view for pivot
df_filtered.createOrReplaceTempView("quality_filtered")

# Pivot — one row per hospital
df_pivoted = spark.sql("""
    SELECT
        provider_id,
        facility_name,
        state,
        MAX(CASE WHEN measure_id = 'MORT_30_AMI'
            THEN score_numeric END)                        AS mort_30_ami,
        MAX(CASE WHEN measure_id = 'MORT_30_HF'
            THEN score_numeric END)                        AS mort_30_hf,
        MAX(CASE WHEN measure_id = 'MORT_30_PN'
            THEN score_numeric END)                        AS mort_30_pn,
        MAX(CASE WHEN measure_id = 'MORT_30_CABG'
            THEN score_numeric END)                        AS mort_30_cabg,
        MAX(CASE WHEN measure_id = 'MORT_30_STK'
            THEN score_numeric END)                        AS mort_30_stk,
        MAX(CASE WHEN measure_id = 'READM_30_AMI'
            THEN score_numeric END)                        AS readm_30_ami,
        MAX(CASE WHEN measure_id = 'READM_30_HF'
            THEN score_numeric END)                        AS readm_30_hf,
        MAX(CASE WHEN measure_id = 'READM_30_PN'
            THEN score_numeric END)                        AS readm_30_pn,
        MAX(CASE WHEN measure_id = 'READM_30_HIP_KNEE'
            THEN score_numeric END)                        AS readm_30_hip_knee,
        MAX(CASE WHEN measure_id = 'PSI_90'
            THEN score_numeric END)                        AS psi_90_safety,
        MAX(CASE WHEN measure_id = 'OP_18b'
            THEN score_numeric END)                        AS op_18b_wait_time,
        MAX(CASE WHEN measure_id = 'EDV'
            THEN score_numeric END)                        AS ed_volume,

        -- Compared to national flags
        MAX(CASE WHEN measure_id = 'MORT_30_HF'
            THEN compared_to_national END)                 AS hf_mortality_vs_national,
        MAX(CASE WHEN measure_id = 'READM_30_HF'
            THEN compared_to_national END)                 AS hf_readmission_vs_national

    FROM quality_filtered
    GROUP BY provider_id, facility_name, state
""")

print(f"Pivoted rows (one per hospital): {df_pivoted.count():,}")
display(df_pivoted.limit(5))


In [0]:
# ============================================================
# STEP 3 — CALCULATE QUALITY SCORE PER HOSPITAL
# Composite quality score 0-100
# ============================================================

print("=" * 55)
print("  CALCULATING QUALITY SCORES")
print("=" * 55)

df_pivoted.createOrReplaceTempView("quality_pivoted")

# National averages for context (approximate CMS benchmarks)
# Mortality rates: lower is better, typical range 10-20%
# Readmission rates: lower is better, typical range 15-25%

df_quality_scored = spark.sql("""
    SELECT
        *,
        -- Average mortality rate across available measures
        ROUND(
            (COALESCE(mort_30_ami, 0) + COALESCE(mort_30_hf, 0) +
             COALESCE(mort_30_pn, 0) + COALESCE(mort_30_stk, 0)) /
            NULLIF(
                (CASE WHEN mort_30_ami IS NOT NULL THEN 1 ELSE 0 END +
                 CASE WHEN mort_30_hf  IS NOT NULL THEN 1 ELSE 0 END +
                 CASE WHEN mort_30_pn  IS NOT NULL THEN 1 ELSE 0 END +
                 CASE WHEN mort_30_stk IS NOT NULL THEN 1 ELSE 0 END),
            0)
        , 2)                                               AS avg_mortality_rate,

        -- Average readmission rate across available measures
        ROUND(
            (COALESCE(readm_30_ami, 0) + COALESCE(readm_30_hf, 0) +
             COALESCE(readm_30_pn, 0) + COALESCE(readm_30_hip_knee, 0)) /
            NULLIF(
                (CASE WHEN readm_30_ami      IS NOT NULL THEN 1 ELSE 0 END +
                 CASE WHEN readm_30_hf       IS NOT NULL THEN 1 ELSE 0 END +
                 CASE WHEN readm_30_pn       IS NOT NULL THEN 1 ELSE 0 END +
                 CASE WHEN readm_30_hip_knee IS NOT NULL THEN 1 ELSE 0 END),
            0)
        , 2)                                               AS avg_readmission_rate,

        -- Quality flags based on comparison to national
        CASE WHEN hf_mortality_vs_national = 'Better than the national rate'
            THEN 1 ELSE 0 END                              AS better_mortality_flag,
        CASE WHEN hf_readmission_vs_national = 'Better than the national rate'
            THEN 1 ELSE 0 END                              AS better_readmission_flag,
        CASE WHEN hf_mortality_vs_national = 'Worse than the national rate'
            THEN 1 ELSE 0 END                              AS worse_mortality_flag,
        CASE WHEN hf_readmission_vs_national = 'Worse than the national rate'
            THEN 1 ELSE 0 END                              AS worse_readmission_flag

    FROM quality_pivoted
""")

print(f"Quality scored rows : {df_quality_scored.count():,}")

print("\nMortality rate distribution:")
spark.sql("""
    SELECT
        ROUND(AVG(avg_mortality_rate), 2) AS national_avg_mortality,
        ROUND(MIN(avg_mortality_rate), 2) AS best_mortality,
        ROUND(MAX(avg_mortality_rate), 2) AS worst_mortality,
        SUM(better_mortality_flag)        AS better_than_national,
        SUM(worse_mortality_flag)         AS worse_than_national
    FROM quality_pivoted q
    JOIN (SELECT provider_id,
            CASE WHEN hf_mortality_vs_national = 'Better than the national rate'
                THEN 1 ELSE 0 END AS better_mortality_flag,
            CASE WHEN hf_mortality_vs_national = 'Worse than the national rate'
                THEN 1 ELSE 0 END AS worse_mortality_flag,
            ROUND(
                (COALESCE(mort_30_ami, 0) + COALESCE(mort_30_hf, 0) +
                 COALESCE(mort_30_pn, 0) + COALESCE(mort_30_stk, 0)) /
                NULLIF(
                    (CASE WHEN mort_30_ami IS NOT NULL THEN 1 ELSE 0 END +
                     CASE WHEN mort_30_hf  IS NOT NULL THEN 1 ELSE 0 END +
                     CASE WHEN mort_30_pn  IS NOT NULL THEN 1 ELSE 0 END +
                     CASE WHEN mort_30_stk IS NOT NULL THEN 1 ELSE 0 END),
                0), 2) AS avg_mortality_rate
          FROM quality_pivoted) s ON q.provider_id = s.provider_id
""").show(truncate=False)

In [0]:
# ============================================================
# STEP 4 — WRITE QUALITY SILVER
# ============================================================

print("=" * 55)
print("  WRITING QUALITY SILVER")
print("=" * 55)

df_quality_scored \
    .withColumn("_batch_id",         lit(BATCH_ID)) \
    .withColumn("_batch_date",        lit(BATCH_DATE)) \
    .withColumn("_silver_processed_at", lit(BATCH_TIMESTAMP)) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_QUALITY_SILVER)

quality_written = spark.table(TBL_QUALITY_SILVER).count()
print(f"Quality Silver rows : {quality_written:,}")
display(spark.table(TBL_QUALITY_SILVER).limit(5))

In [0]:
# ============================================================
# STEP 5 — TRANSFORM HCAHPS
# Extract key satisfaction scores per hospital
# ============================================================

print("=" * 55)
print("  TRANSFORMING HCAHPS SCORES")
print("=" * 55)

# Key HCAHPS measures
df_hcahps_raw = spark.table(TBL_HCAHPS_RAW)

print(f"HCAHPS raw rows : {df_hcahps_raw.count():,}")
print("\nUnique measure IDs:")
spark.sql(f"""
    SELECT DISTINCT hcahps_measure_id, hcahps_question
    FROM {TBL_HCAHPS_RAW}
    WHERE hcahps_measure_id LIKE '%STAR%'
    OR hcahps_measure_id LIKE '%RECMND%'
    OR hcahps_measure_id LIKE '%LINEAR%'
    ORDER BY hcahps_measure_id
""").show(50, truncate=False)

In [0]:
# ============================================================
# STEP 6 — PIVOT HCAHPS AND WRITE (corrected measure IDs)
# ============================================================

print("=" * 55)
print("  PIVOTING AND WRITING HCAHPS")
print("=" * 55)

df_hcahps_pivoted = spark.sql(f"""
    SELECT
        facility_id                                        AS provider_id,
        facility_name,
        state,
        number_of_completed_surveys,
        survey_response_rate_percent,

        -- Overall star rating
        MAX(CASE WHEN hcahps_measure_id = 'H_STAR_RATING'
            THEN TRY_CAST(patient_survey_star_rating AS DOUBLE) END) AS overall_star_rating,

        -- Domain star ratings
        MAX(CASE WHEN hcahps_measure_id = 'H_COMP_1_STAR_RATING'
            THEN TRY_CAST(patient_survey_star_rating AS DOUBLE) END) AS nurse_star_rating,
        MAX(CASE WHEN hcahps_measure_id = 'H_COMP_2_STAR_RATING'
            THEN TRY_CAST(patient_survey_star_rating AS DOUBLE) END) AS doctor_star_rating,
        MAX(CASE WHEN hcahps_measure_id = 'H_CLEAN_STAR_RATING'
            THEN TRY_CAST(patient_survey_star_rating AS DOUBLE) END) AS cleanliness_star_rating,
        MAX(CASE WHEN hcahps_measure_id = 'H_QUIET_STAR_RATING'
            THEN TRY_CAST(patient_survey_star_rating AS DOUBLE) END) AS quietness_star_rating,
        MAX(CASE WHEN hcahps_measure_id = 'H_COMP_5_STAR_RATING'
            THEN TRY_CAST(patient_survey_star_rating AS DOUBLE) END) AS medication_comm_star_rating,
        MAX(CASE WHEN hcahps_measure_id = 'H_COMP_6_STAR_RATING'
            THEN TRY_CAST(patient_survey_star_rating AS DOUBLE) END) AS discharge_info_star_rating,
        MAX(CASE WHEN hcahps_measure_id = 'H_RECMND_STAR_RATING'
            THEN TRY_CAST(patient_survey_star_rating AS DOUBLE) END) AS recommend_star_rating,

        -- Recommend hospital percent
        MAX(CASE WHEN hcahps_measure_id = 'H_RECMND_DY'
            THEN TRY_CAST(hcahps_answer_percent AS DOUBLE) END)      AS pct_definitely_recommend,
        MAX(CASE WHEN hcahps_measure_id = 'H_RECMND_PY'
            THEN TRY_CAST(hcahps_answer_percent AS DOUBLE) END)      AS pct_probably_recommend,
        MAX(CASE WHEN hcahps_measure_id = 'H_RECMND_DN'
            THEN TRY_CAST(hcahps_answer_percent AS DOUBLE) END)      AS pct_not_recommend,

        -- Linear scores (0-100)
        MAX(CASE WHEN hcahps_measure_id = 'H_COMP_1_LINEAR_SCORE'
            THEN TRY_CAST(hcahps_linear_mean_value AS DOUBLE) END)   AS nurse_linear_score,
        MAX(CASE WHEN hcahps_measure_id = 'H_COMP_2_LINEAR_SCORE'
            THEN TRY_CAST(hcahps_linear_mean_value AS DOUBLE) END)   AS doctor_linear_score,
        MAX(CASE WHEN hcahps_measure_id = 'H_CLEAN_LINEAR_SCORE'
            THEN TRY_CAST(hcahps_linear_mean_value AS DOUBLE) END)   AS cleanliness_linear_score,
        MAX(CASE WHEN hcahps_measure_id = 'H_QUIET_LINEAR_SCORE'
            THEN TRY_CAST(hcahps_linear_mean_value AS DOUBLE) END)   AS quietness_linear_score,
        MAX(CASE WHEN hcahps_measure_id = 'H_RECMND_LINEAR_SCORE'
            THEN TRY_CAST(hcahps_linear_mean_value AS DOUBLE) END)   AS recommend_linear_score,
        MAX(CASE WHEN hcahps_measure_id = 'H_HSP_RATING_LINEAR_SCORE'
            THEN TRY_CAST(hcahps_linear_mean_value AS DOUBLE) END)   AS overall_rating_linear_score

    FROM {TBL_HCAHPS_RAW}
    GROUP BY
        facility_id,
        facility_name,
        state,
        number_of_completed_surveys,
        survey_response_rate_percent
""")

df_hcahps_pivoted \
    .withColumn("_batch_id",             lit(BATCH_ID)) \
    .withColumn("_batch_date",           lit(BATCH_DATE)) \
    .withColumn("_silver_processed_at",  lit(BATCH_TIMESTAMP)) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_HCAHPS_SILVER)

hcahps_written = spark.table(TBL_HCAHPS_SILVER).count()
print(f"HCAHPS Silver rows : {hcahps_written:,}")

print("\nSample HCAHPS scores:")
display(
    spark.table(TBL_HCAHPS_SILVER)
        .select(
            "provider_id",
            "facility_name",
            "state",
            "overall_star_rating",
            "pct_definitely_recommend",
            "nurse_linear_score",
            "doctor_linear_score",
            "recommend_linear_score",
            "overall_rating_linear_score"
        )
        .filter(col("overall_star_rating").isNotNull())
        .limit(10)
)

In [0]:
# ============================================================
# STEP 7 — LOG TO AUDIT
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "silver",
    status           = "SUCCESS",
    rows_read        = df_quality_raw.count() + df_hcahps_raw.count(),
    rows_written     = quality_written + hcahps_written,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"quality scores: {quality_written:,} hospitals | "
        f"HCAHPS scores: {hcahps_written:,} hospitals"
    )
)